<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/04_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 04_results_and_evaluation.ipynb
# Pure analysis notebook — no model training.
# Loads all outputs saved by 03b_survival_modeling.ipynb.
# ============================================================
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    calibration_curve,
)
from lifelines import KaplanMeierFitter
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/mimic_iv_processed')
FIG_DIR    = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

HORIZONS  = [6, 12, 24]
NUM_BINS  = 48
MAX_HOURS = 200
time_cuts = np.linspace(0, MAX_HOURS, NUM_BINS + 1)[1:]

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {name}.png ✓")

print("Imports complete ✓")

In [ ]:
# ============================================================
# 04_results_and_evaluation.ipynb
# Pure analysis notebook — no model training.
# Loads all outputs saved by 03b_survival_modeling.ipynb.
# ============================================================
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    calibration_curve,
)
from lifelines import KaplanMeierFitter
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/mimic_iv_processed')
FIG_DIR    = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

HORIZONS  = [6, 12, 24]
NUM_BINS  = 48
MAX_HOURS = 200
time_cuts = np.linspace(0, MAX_HOURS, NUM_BINS + 1)[1:]

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {name}.png ✓")

print("Imports complete ✓")

## 2. Canonical Results Table

In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print("=" * 100)
print("COMPLETE RESULTS — Phase 2 Survival Modeling")
print("=" * 100)
print(results_df.to_string(index=False))
print()

# Best per metric per dataset
for dataset in ['Incident', 'All Sepsis']:
    sub = results_df[results_df['Dataset'] == dataset].copy()
    if sub.empty:
        continue
    print(f"── Best per metric ({dataset}) ─────────────────────────")
    print(f"  C-index : "
          f"{sub.loc[sub['C-index'].idxmax(), 'Model']} "
          f"({sub.loc[sub['C-index'].idxmax(), 'Stage']}) "
          f"→ {sub['C-index'].max():.4f}")
    print(f"  IBS     : "
          f"{sub.loc[sub['IBS'].idxmin(), 'Model']} "
          f"({sub.loc[sub['IBS'].idxmin(), 'Stage']}) "
          f"→ {sub['IBS'].min():.4f}")
    print()

## 3. Calibration Curves — All Models, Both Datasets

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    'Calibration Curves at Fixed Horizons\n'
    'Before vs After Isotonic Regression Calibration',
    fontsize=13, fontweight='bold'
)

configs = [
    ('DDH — Incident',     surv_ir,  surv_ic,  dur_i,  evt_i),
    ('DDH — All Sepsis',   surv_ar,  surv_ac,  dur_a,  evt_a),
    ('Transformer — Inc.', surv_tir, surv_tic, dur_ti, evt_ti),
]
check_horizons = [6, 24]

for col_idx, (label, surv_raw, surv_cal, durs, evts) in enumerate(configs):
    for row_idx, H in enumerate(check_horizons):
        ax     = axes[row_idx, col_idx]
        bin_H  = int(np.clip(
            np.searchsorted(time_cuts, H, side='right'),
            0, NUM_BINS - 1
        ))
        y_true = ((evts == 1) & (durs <= H)).astype(int)

        if y_true.sum() < 5:
            ax.text(0.5, 0.5, 'Insufficient events',
                    ha='center', va='center',
                    transform=ax.transAxes)
            continue

        for surv_arr, lbl, style, clr in [
            (surv_raw, 'Before cal', '--', 'tomato'),
            (surv_cal, 'After cal',  '-',  'steelblue'),
        ]:
            y_sc = (1 - surv_arr[:, bin_H]).clip(0, 1)
            frac, mean = calibration_curve(
                y_true, y_sc, n_bins=8, strategy='uniform'
            )
            ax.plot(mean, frac, style, color=clr,
                    lw=2, label=lbl)

        ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Perfect')
        ax.set_title(f'{label}\nt={H}h  (pos={y_true.sum():,})')
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Fraction of positives')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
savefig('calibration_all_models')

## 4. AUROC / AUPRC at Clinical Horizons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'AUROC Comparison at Clinical Horizons\n'
    'All models — Incident Sepsis (calibrated)',
    fontsize=13, fontweight='bold'
)

model_labels = ['DeepSurv', 'DDH (calibrated)', 'Transformer (cal)']
bar_colors   = ['#607D8B', '#F44336', '#2196F3']
x = np.arange(len(model_labels))

for col_idx, H in enumerate(HORIZONS):
    ax   = axes[col_idx]
    vals = []
    for m in model_labels:
        key = f"{m}|Incident"
        if key in horizon_results:
            vals.append(horizon_results[key][H]['auroc'])
        else:
            vals.append(float('nan'))

    bars = ax.bar(x, vals, 0.6,
                  color=bar_colors, alpha=0.87)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f'{v:.3f}',
                ha='center', va='bottom',
                fontsize=10, fontweight='bold'
            )

    ax.set_xticks(x)
    ax.set_xticklabels(model_labels,
                       rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('AUROC')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'AUROC @ {H}h')
    ax.axhline(0.5, color='black', lw=1,
               ls='--', alpha=0.4, label='Random')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
savefig('auroc_comparison')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    'ROC and PR Curves at Clinical Horizons\n'
    'DDH (calibrated) — Incident Sepsis',
    fontsize=13, fontweight='bold'
)

colors = {6: '#F44336', 12: '#FF9800', 24: '#2196F3'}

for col_idx, H in enumerate(HORIZONS):
    bin_H  = int(np.clip(
        np.searchsorted(time_cuts, H, side='right'),
        0, NUM_BINS - 1
    ))
    y_true  = ((evt_i == 1) & (dur_i <= H)).astype(int)
    y_score = (1 - surv_ic[:, bin_H]).clip(0, 1)

    if y_true.sum() < 5:
        for row in range(2):
            axes[row, col_idx].text(
                0.5, 0.5, 'Too few events',
                ha='center', va='center',
                transform=axes[row, col_idx].transAxes
            )
        continue

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc       = roc_auc_score(y_true, y_score)
    axes[0, col_idx].plot(fpr, tpr, color=colors[H],
                          lw=2.5,
                          label=f'AUROC={auroc:.3f}')
    axes[0, col_idx].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    axes[0, col_idx].set_xlabel('FPR')
    axes[0, col_idx].set_ylabel('TPR')
    axes[0, col_idx].set_title(
        f'ROC @ {H}h\n'
        f'(pos={y_true.sum():,}/{len(y_true):,})'
    )
    axes[0, col_idx].legend(fontsize=9)
    axes[0, col_idx].grid(alpha=0.3)

    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    auprc        = average_precision_score(y_true, y_score)
    baseline     = y_true.mean()
    axes[1, col_idx].plot(rec, prec, color=colors[H],
                          lw=2.5,
                          label=f'AUPRC={auprc:.3f}')
    axes[1, col_idx].axhline(
        baseline, color='gray', lw=1, ls='--',
        label=f'Baseline={baseline:.3f}'
    )
    axes[1, col_idx].set_xlabel('Recall')
    axes[1, col_idx].set_ylabel('Precision')
    axes[1, col_idx].set_title(f'PR @ {H}h')
    axes[1, col_idx].legend(fontsize=9)
    axes[1, col_idx].grid(alpha=0.3)

plt.tight_layout()
savefig('roc_pr_horizons')

## 5. Predicted Survival vs Kaplan-Meier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Predicted Survival vs Kaplan-Meier\n'
    'Incident Sepsis — Test Set',
    fontsize=13, fontweight='bold'
)

for ax, title, surv_arr, durs, evts in [
    (axes[0], 'DDH (calibrated)',         surv_ic,  dur_i,  evt_i),
    (axes[1], 'Transformer (calibrated)', surv_tic, dur_ti, evt_ti),
]:
    # Kaplan-Meier
    kmf = KaplanMeierFitter()
    kmf.fit(durs, event_observed=evts,
            label='Kaplan-Meier (observed)')
    kmf.plot_survival_function(ax=ax, color='black',
                               lw=2.5, ci_show=True)

    # Mean predicted survival
    mean_surv = surv_arr.mean(axis=0)
    ax.plot(time_cuts, mean_surv,
            color='tomato', lw=2.5, ls='--',
            label='Mean predicted S(t)')

    # 10th–90th percentile band
    p10 = np.percentile(surv_arr, 10, axis=0)
    p90 = np.percentile(surv_arr, 90, axis=0)
    ax.fill_between(time_cuts, p10, p90,
                    alpha=0.15, color='tomato',
                    label='10th–90th pct')

    ax.set_xlabel('Hours since ICU admission')
    ax.set_ylabel('Survival probability (sepsis-free)')
    ax.set_title(title)
    ax.set_xlim(0, 150)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
savefig('survival_km_vs_model')

## 6. SpO2 Dominance — Diagnosis Figure

In [ ]:
# Load winsorised scaler and bounds
scaler_w = joblib.load(str(OUTPUT_DIR / 'survival_scaler_winsorised.pkl'))
lo2      = np.load(str(OUTPUT_DIR / 'winsor_lo.npy'))
hi2      = np.load(str(OUTPUT_DIR / 'winsor_hi.npy'))

# Load original scaler
scaler_orig = joblib.load(
    str(OUTPUT_DIR / 'survival_scaler_incident.pkl')
)

# Rebuild test features to compare variance
surv_inc_data = pd.read_csv(
    OUTPUT_DIR / 'survival_incident_dataset.csv'
)
all_features  = pd.read_csv(OUTPUT_DIR / 'engineered_features.csv')
test_ids      = surv_inc_data[
    surv_inc_data['split'] == 'test'
]['stay_id']
test_raw = all_features[
    all_features['stay_id'].isin(test_ids)
][feature_cols].fillna(0).values

X_orig = scaler_orig.transform(test_raw)
X_wsor = scaler_w.transform(np.clip(test_raw, lo2, hi2))

var_orig = X_orig.var(axis=0)
var_wsor = X_wsor.var(axis=0)

# SHAP importance
mabs_orig = np.abs(shap_values).mean(axis=(0, 2))
top15_orig = np.argsort(mabs_orig)[::-1][:15]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(
    'SpO2 Dominance — Diagnosis and Fix\n'
    'Left: original SHAP | '
    'Middle: variance comparison | '
    'Right: would update after winsorised SHAP re-run',
    fontsize=12, fontweight='bold'
)

# Original SHAP importance
spo2_mask = ['spo2' in feature_cols[i].lower()
             for i in top15_orig]
bar_colors_orig = ['tomato' if m else 'steelblue'
                   for m in spo2_mask]
axes[0].barh(
    range(15),
    mabs_orig[top15_orig][::-1],
    color=bar_colors_orig[::-1], alpha=0.85
)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(
    [feature_cols[i] for i in top15_orig][::-1], fontsize=8
)
axes[0].set_xlabel('Mean |SHAP|')
axes[0].set_title('Original SHAP\n(red = SpO2 features)')
axes[0].grid(axis='x', alpha=0.3)

# Variance before vs after
top15_var = np.argsort(var_orig)[::-1][:15]
x_pos     = np.arange(15)
axes[1].bar(x_pos - 0.2, var_orig[top15_var], 0.4,
            color='tomato', alpha=0.8, label='Original')
axes[1].bar(x_pos + 0.2, var_wsor[top15_var], 0.4,
            color='steelblue', alpha=0.8, label='Winsorised')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(
    [feature_cols[i] for i in top15_var],
    rotation=45, ha='right', fontsize=7
)
axes[1].set_ylabel('Scaled feature variance')
axes[1].set_title('Feature Variance\nBefore vs After Winsorisation')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Placeholder for winsorised SHAP
axes[2].text(
    0.5, 0.5,
    'Re-run SHAP on winsorised\nfeatures in 03b to populate\n'
    'this panel.\n\nExpected: SpO2 features\nno longer dominate.',
    ha='center', va='center',
    fontsize=11, color='gray',
    transform=axes[2].transAxes
)
axes[2].set_title('Winsorised SHAP\n(pending re-run)')
axes[2].grid(alpha=0.2)

plt.tight_layout()
savefig('spo2_dominance_diagnosis')

## 7. SurvSHAP(t) — Feature Importance

In [ ]:
mabs      = np.abs(shap_values).mean(axis=(0, 2))
top15_idx = np.argsort(mabs)[::-1][:15]
top15_names = [feature_cols[i] for i in top15_idx]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle(
    'SurvSHAP(t) — Feature Importance\n'
    'Dynamic-DeepHit (Incident Sepsis)',
    fontsize=13, fontweight='bold'
)

# Bar chart
axes[0].barh(
    range(15),
    mabs[top15_idx][::-1],
    color=cm.RdYlGn(np.linspace(0.8, 0.2, 15))[::-1],
    alpha=0.87
)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(top15_names[::-1], fontsize=9)
axes[0].set_xlabel('Mean |SHAP| (averaged over time points)')
axes[0].set_title('Top 15 Features — Overall Importance')
axes[0].grid(axis='x', alpha=0.3)

# Heatmap — importance over time
n_times         = shap_values.shape[2]
shap_time_top   = np.abs(shap_values).mean(axis=0)[top15_idx]
im = axes[1].imshow(
    shap_time_top, aspect='auto', cmap='YlOrRd'
)
axes[1].set_xticks(range(n_times))
axes[1].set_xticklabels(
    [f'{t:.0f}h' for t in shap_times],
    rotation=45, fontsize=7
)
axes[1].set_yticks(range(15))
axes[1].set_yticklabels(top15_names, fontsize=8)
axes[1].set_xlabel('Hours since ICU admission')
axes[1].set_title('Importance Over Time')
plt.colorbar(im, ax=axes[1], label='Mean |SHAP|')

plt.tight_layout()
savefig('shap_importance')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    'SurvSHAP(t) — Direction of Effect Over Time\n'
    'Positive = increases sepsis-free survival  |  '
    'Negative = increases sepsis risk',
    fontsize=12, fontweight='bold'
)

top5_idx    = top15_idx[:5]
palette     = ['#F44336','#2196F3','#4CAF50','#FF9800','#9C27B0']
signed_shap = shap_values.mean(axis=0)   # (n_features, n_times)

for i, (fi, clr) in enumerate(zip(top5_idx, palette)):
    label = feature_cols[fi]
    axes[0].plot(shap_times,
                 np.abs(shap_values[:, fi, :]).mean(axis=0),
                 color=clr, lw=2.5, marker='o', ms=5,
                 label=label)
    axes[1].plot(shap_times,
                 signed_shap[fi],
                 color=clr, lw=2.5, marker='o', ms=5,
                 label=label)

axes[0].set_xlabel('Hours since ICU admission')
axes[0].set_ylabel('Mean |SHAP|')
axes[0].set_title('Magnitude Over Time')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color='black', lw=1, ls='--', alpha=0.5)
axes[1].set_xlabel('Hours since ICU admission')
axes[1].set_ylabel('Mean SHAP')
axes[1].set_title('Direction Over Time')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
savefig('shap_direction')

## 8. Model Comparison Summary

In [ ]:
fig = plt.figure(figsize=(20, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        hspace=0.4, wspace=0.35)
fig.suptitle(
    'Phase 2 — Model Comparison Summary\n'
    'Incident Sepsis (primary) | All Sepsis (sensitivity)',
    fontsize=14, fontweight='bold'
)

model_order  = ['DeepSurv', 'DDH', 'Transformer']
colors_map   = {
    'DeepSurv'   : '#607D8B',
    'DDH'        : '#F44336',
    'Transformer': '#2196F3',
}
x = np.arange(len(model_order))

def bar_subplot(ax, metric, dataset, title, ylim=(0, 1)):
    vals = []
    for m in model_order:
        # Use calibrated stage where available
        sub = results_df[
            (results_df['Model'] == m) &
            (results_df['Dataset'] == dataset) &
            (results_df['Stage'].isin(['Calibrated', '—']))
        ]
        if len(sub) == 0:
            vals.append(0.0)
        else:
            vals.append(float(sub[metric].iloc[0])
                        if metric in sub.columns
                        else 0.0)

    bars = ax.bar(x, vals, 0.6,
                  color=[colors_map[m] for m in model_order],
                  alpha=0.87)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{v:.3f}',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold'
        )
    ax.set_xticks(x)
    ax.set_xticklabels(model_order,
                       rotation=15, ha='right', fontsize=9)
    ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

bar_subplot(fig.add_subplot(gs[0, 0]),
            'C-index', 'Incident',
            'C-index — Incident', (0.5, 1))
bar_subplot(fig.add_subplot(gs[0, 1]),
            'C-index', 'All Sepsis',
            'C-index — All Sepsis', (0.5, 1))
bar_subplot(fig.add_subplot(gs[0, 2]),
            'IBS', 'Incident',
            'IBS — Incident (lower is better)', (0, 0.3))
bar_subplot(fig.add_subplot(gs[1, 0]),
            'AUROC@6h', 'Incident',
            'AUROC@6h — Incident', (0.5, 1))
bar_subplot(fig.add_subplot(gs[1, 1]),
            'AUROC@12h', 'Incident',
            'AUROC@12h — Incident', (0.5, 1))
bar_subplot(fig.add_subplot(gs[1, 2]),
            'AUROC@24h', 'Incident',
            'AUROC@24h — Incident', (0.5, 1))

savefig('model_comparison_summary')

print("\n" + "=" * 60)
print("04_results_and_evaluation.ipynb COMPLETE")
print("=" * 60)
print(f"\nAll figures saved to: {FIG_DIR}")